# Phase 5 — SVM Experiments
Goal: test whether margin-based models (linear, then RBF) beat the Logistic Regression baseline on expected APS cost.
Baseline to beat (honest, fold-wise): balanced LogReg + signed-log = 51,930.

In [1]:
%load_ext autoreload
%autoreload 2

import sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))

from src.data_loader import load_aps_data
from src.preprocessing import build_linear_preprocessor, make_cv_splitter
from src.evaluation import best_cost_threshold

DATA_DIR = PROJECT_ROOT / "dataset"
X,y = load_aps_data(DATA_DIR/"aps_failure_training_set.csv")
cv = make_cv_splitter()
print(X.shape, dict(y.value_counts()))

(60000, 170) {0: np.int64(59000), 1: np.int64(1000)}


In [3]:
from numpy import size
rng = np.random.default_rng(0)
y_demo = rng.integers(0,2,2000)
s_demo = rng.normal(size=2000) + 3*y_demo-1.5
print(best_cost_threshold(y_demo,s_demo))

{'tn': 653, 'fp': 304, 'fn': 2, 'tp': 1041, 'aps_cost': 4040, 'threshold': -1.0890802948302896, 'cost_per_truck': 2.02}


In [4]:
from numpy import record
from matplotlib import use
def make_linear_svm(C,class_weight):
    return Pipeline([
        ("prep", build_linear_preprocessor(use_log=True, drop_duplicates=True)),
        ("svm", LinearSVC(C=C, class_weight=class_weight, dual=False, max_iter=2000, random_state=42))
    ])

def oof_scores(model, X,y,cv):
    """Out-of-fold decision scores. Each row is scored by a model that never saw it."""
    scores = np.zeros(len(y))
    n_warn, t0=0, time.time()
    for tr, va in cv.split(X,y):
        m = clone(model)
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            m.fit(X.iloc[tr], y.iloc[tr])
        n_warn += any(issubclass(x.category, ConvergenceWarning) for x in w)
        scores[va] = m.decision_function(X.iloc[va])
    return scores, n_warn, time.time() - t0


def honest_cost(y, scores, cv):
    """Per fold: pick the threshold on the OTHER folds' scores, score it on this fold."""
    y_arr = np.asarray(y)
    total, FP, FN = 0, 0, 0
    for tr, va in cv.split(np.zeros(len(y_arr)), y_arr):
        t = best_cost_threshold(y_arr[tr], scores[tr])["threshold"]
        pred = scores[va] >= t
        fp = int((pred & (y_arr[va] == 0)).sum())
        fn = int((~pred & (y_arr[va] == 1)).sum())
        FP, FN, total = FP + fp, FN + fn, total + 10 * fp + 500 * fn
    return total, FP, FN

In [5]:
s, n_warn, secs = oof_scores(make_linear_svm(1.0, "balanced"), X, y, cv)
print("warn folds:", n_warn, "| seconds:", round(secs, 1))
print("ROC-AUC:", round(roc_auc_score(y, s), 4), "| PR-AUC:", round(average_precision_score(y, s), 4))
print(best_cost_threshold(y, s))
print("honest:", honest_cost(y, s, cv))

warn folds: 0 | seconds: 158.6
ROC-AUC: 0.9827 | PR-AUC: 0.7747
{'tn': 56895, 'fp': 2105, 'fn': 62, 'tp': 938, 'aps_cost': 52050, 'threshold': -0.03505789149988825, 'cost_per_truck': 0.8675}
honest: (54640, 2114, 67)


In [6]:
all_scores = {}   # (C, class_weight) -> out-of-fold scores

extra = [(C, None) for C in [3.0, 10.0, 30.0, 100.0]] + \
        [(C, "balanced") for C in [0.001, 0.003]]

for C, cw in extra:
    s, n_warn, secs = oof_scores(make_linear_svm(C, cw), X, y, cv)
    all_scores[(C, cw)] = s
    tuned = best_cost_threshold(y, s)
    h_cost, h_fp, h_fn = honest_cost(y, s, cv)
    rows.append({"model": "LinearSVC", "C": C, "class_weight": cw,
                 "warn_folds": n_warn, "seconds": round(secs, 1),
                 "roc_auc": roc_auc_score(y, s),
                 "pr_auc": average_precision_score(y, s),
                 "tuned_cost": tuned["aps_cost"],
                 "honest_cost": h_cost, "honest_fp": h_fp, "honest_fn": h_fn})

svm_linear_results = pd.DataFrame(rows).sort_values("honest_cost").reset_index(drop=True)
svm_linear_results.round(4)

NameError: name 'rows' is not defined

In [ ]:
svm_linear_results.to_csv(PROJECT_ROOT / "reports" / "results" / "phase5_linear_svm_results.csv", index=False)

warn folds: 0 | seconds: 135.7
ROC-AUC: 0.9827 | PR-AUC: 0.7747
{'tn': 56895, 'fp': 2105, 'fn': 62, 'tp': 938, 'aps_cost': 52050, 'threshold': -0.03505789149988825, 'cost_per_truck': 0.8675}
honest: (54640, 2114, 67)


In [ ]:
from sklearn.linear_model import LogisticRegression

def make_logreg(class_weight):
    return Pipeline([
        ("prep", build_linear_preprocessor(use_log=True, drop_duplicates=True)),
        ("lr", LogisticRegression(max_iter=500, solver="lbfgs", tol=1e-3,
                                  random_state=42, class_weight=class_weight)),
    ])

def fold_costs(y, scores, cv):
    """Honest cost for each fold separately."""
    y_arr, out = np.asarray(y), []
    for tr, va in cv.split(np.zeros(len(y_arr)), y_arr):
        t = best_cost_threshold(y_arr[tr], scores[tr])["threshold"]
        pred = scores[va] >= t
        fp = int((pred & (y_arr[va] == 0)).sum())
        fn = int((~pred & (y_arr[va] == 1)).sum())
        out.append(10 * fp + 500 * fn)
    return np.array(out)

svm_s, _, _ = oof_scores(make_linear_svm(1.0, None), X, y, cv)
lr_u, _, _  = oof_scores(make_logreg(None), X, y, cv)
lr_b, _, _  = oof_scores(make_logreg("balanced"), X, y, cv)

fc = pd.DataFrame({
    "SVM C=1 unweighted": fold_costs(y, svm_s, cv),
    "LogReg unweighted":  fold_costs(y, lr_u, cv),
    "LogReg balanced":    fold_costs(y, lr_b, cv),
}, index=[f"fold {i}" for i in range(1, 6)])
fc.loc["total"] = fc.sum()
fc

,SVM C=1 unweighted,LogReg unweighted,LogReg balanced
fold 1,8920,8050,9820
fold 2,10650,10830,12070
fold 3,8480,9660,9350
fold 4,8420,10030,11100
fold 5,8630,10180,9590
total,45100,48750,51930


In [7]:
from sklearn.kernel_approximation import Nystroem

N_FEATURES = 225   # features after preprocessing (duplicates dropped, log on)

def make_rbf_svm(C, gamma_mult, n_components=300, class_weight=None):
    return Pipeline([
        ("prep", build_linear_preprocessor(use_log=True, drop_duplicates=True)),
        ("nys", Nystroem(kernel="rbf", gamma=gamma_mult / N_FEATURES,
                         n_components=n_components, random_state=42)),
        ("svm", LinearSVC(C=C, class_weight=class_weight, dual=False,
                          max_iter=2000, random_state=42)),
    ])

# 1) time ONE fold first
tr, va = next(iter(cv.split(X, y)))
t0 = time.time()
make_rbf_svm(C=1.0, gamma_mult=0.3).fit(X.iloc[tr], y.iloc[tr])
print("one fold fit:", round(time.time() - t0, 1), "seconds")

# 2) full out-of-fold run for this one configuration
s, n_warn, secs = oof_scores(make_rbf_svm(C=1.0, gamma_mult=0.3), X, y, cv)
print("warn folds:", n_warn, "| seconds:", round(secs, 1))
print("ROC-AUC:", round(roc_auc_score(y, s), 4), "| PR-AUC:", round(average_precision_score(y, s), 4))
print("tuned:", best_cost_threshold(y, s)["aps_cost"], "| honest:", honest_cost(y, s, cv))

one fold fit: 9.3 seconds
warn folds: 0 | seconds: 53.4
ROC-AUC: 0.9808 | PR-AUC: 0.714
tuned: 57480 | honest: (58720, 2672, 64)


In [8]:
rbf_rows = []

for n_comp in [300, 1000]:
    for g in [0.01, 0.03, 0.1]:
        s, n_warn, secs = oof_scores(
            make_rbf_svm(C=1.0, gamma_mult=g, n_components=n_comp), X, y, cv)
        tuned = best_cost_threshold(y, s)
        h_cost, h_fp, h_fn = honest_cost(y, s, cv)
        rbf_rows.append({"model": "RBF-Nystroem+LinearSVC", "gamma_mult": g,
                         "n_components": n_comp, "C": 1.0,
                         "warn_folds": n_warn, "seconds": round(secs, 1),
                         "roc_auc": roc_auc_score(y, s),
                         "pr_auc": average_precision_score(y, s),
                         "tuned_cost": tuned["aps_cost"],
                         "honest_cost": h_cost, "honest_fp": h_fp, "honest_fn": h_fn})
        print(rbf_rows[-1]["gamma_mult"], n_comp, "->", h_cost, f"({secs:.0f}s)")

rbf_results = pd.DataFrame(rbf_rows).sort_values("honest_cost").reset_index(drop=True)
rbf_results.round(4)

0.01 300 -> 69600 (43s)
0.03 300 -> 57600 (60s)
0.1 300 -> 55300 (56s)
0.01 1000 -> 66950 (88s)
0.03 1000 -> 53090 (88s)
0.1 1000 -> 48780 (84s)


,model,gamma_mult,n_components,C,warn_folds,seconds,roc_auc,pr_auc,tuned_cost,honest_cost,honest_fp,honest_fn
0,RBF-Nystroem+LinearSVC,0.10,1000,1.0,0,84.2,0.9840,0.7765,47920,48780,1978,58
1,RBF-Nystroem+LinearSVC,0.03,1000,1.0,0,87.7,0.9827,0.7482,51710,53090,2959,47
2,RBF-Nystroem+LinearSVC,0.10,300,1.0,0,56.0,0.9831,0.7482,53370,55300,2280,65
3,RBF-Nystroem+LinearSVC,0.03,300,1.0,0,60.0,0.9818,0.7346,55940,57600,3360,48
4,RBF-Nystroem+LinearSVC,0.01,1000,1.0,0,88.5,0.9798,0.7125,64260,66950,3895,56
5,RBF-Nystroem+LinearSVC,0.01,300,1.0,0,42.8,0.9793,0.7073,65730,69600,3810,63


In [9]:
runs = [  # (C, gamma_mult, n_components)
    (1.0, 0.3, 1000),    # finish the gamma curve at 1000 landmarks
    (1.0, 1.0, 1000),
    (10.0, 0.03, 1000),  # test: is low-gamma really under-regularized?
    (10.0, 0.1, 1000),
]

more_rows = []
for C, g, n_comp in runs:
    s, n_warn, secs = oof_scores(
        make_rbf_svm(C=C, gamma_mult=g, n_components=n_comp), X, y, cv)
    tuned = best_cost_threshold(y, s)
    h_cost, h_fp, h_fn = honest_cost(y, s, cv)
    more_rows.append({"model": "RBF-Nystroem+LinearSVC", "gamma_mult": g,
                      "n_components": n_comp, "C": C,
                      "warn_folds": n_warn, "seconds": round(secs, 1),
                      "roc_auc": roc_auc_score(y, s),
                      "pr_auc": average_precision_score(y, s),
                      "tuned_cost": tuned["aps_cost"],
                      "honest_cost": h_cost, "honest_fp": h_fp, "honest_fn": h_fn})
    print(f"C={C} g={g} m={n_comp} -> {h_cost} ({secs:.0f}s, warn={n_warn})")

rbf_results = (pd.concat([rbf_results, pd.DataFrame(more_rows)], ignore_index=True)
                 .sort_values("honest_cost").reset_index(drop=True))
rbf_results.round(4)

C=1.0 g=0.3 m=1000 -> 54520 (117s, warn=0)
C=1.0 g=1.0 m=1000 -> 59050 (178s, warn=0)
C=10.0 g=0.03 m=1000 -> 47550 (164s, warn=0)
C=10.0 g=0.1 m=1000 -> 50990 (122s, warn=0)


,model,gamma_mult,n_components,C,warn_folds,seconds,roc_auc,pr_auc,tuned_cost,honest_cost,honest_fp,honest_fn
0,RBF-Nystroem+LinearSVC,0.03,1000,10.0,0,163.6,0.9846,0.8061,46280,47550,2005,55
1,RBF-Nystroem+LinearSVC,0.10,1000,1.0,0,84.2,0.9840,0.7765,47920,48780,1978,58
2,RBF-Nystroem+LinearSVC,0.10,1000,10.0,0,122.1,0.9828,0.8126,49230,50990,1899,64
3,RBF-Nystroem+LinearSVC,0.03,1000,1.0,0,87.7,0.9827,0.7482,51710,53090,2959,47
4,RBF-Nystroem+LinearSVC,0.30,1000,1.0,0,117.3,0.9824,0.7691,52150,54520,2252,64
5,RBF-Nystroem+LinearSVC,0.10,300,1.0,0,56.0,0.9831,0.7482,53370,55300,2280,65
6,RBF-Nystroem+LinearSVC,0.03,300,1.0,0,60.0,0.9818,0.7346,55940,57600,3360,48
7,RBF-Nystroem+LinearSVC,1.00,1000,1.0,0,178.0,0.9757,0.6701,58550,59050,2555,67
8,RBF-Nystroem+LinearSVC,0.01,1000,1.0,0,88.5,0.9798,0.7125,64260,66950,3895,56
9,RBF-Nystroem+LinearSVC,0.01,300,1.0,0,42.8,0.9793,0.7073,65730,69600,3810,63


In [10]:
from sklearn.pipeline import FeatureUnion

def make_linear_plus_rbf(C, gamma_mult, n_components=1000):
    return Pipeline([
        ("prep", build_linear_preprocessor(use_log=True, drop_duplicates=True)),
        ("features", FeatureUnion([
            ("orig", "passthrough"),           # keep all 225 original features
            ("nys", Nystroem(kernel="rbf", gamma=gamma_mult / N_FEATURES,
                             n_components=n_components, random_state=42)),
        ])),
        ("svm", LinearSVC(C=C, dual=False, max_iter=2000, random_state=42)),
    ])

combo_rows = []
for C, g in [(1.0, 0.1), (1.0, 0.03), (10.0, 0.03)]:
    s, n_warn, secs = oof_scores(make_linear_plus_rbf(C, g), X, y, cv)
    tuned = best_cost_threshold(y, s)
    h_cost, h_fp, h_fn = honest_cost(y, s, cv)
    combo_rows.append({"model": "Linear+RBF-Nystroem", "gamma_mult": g,
                       "n_components": 1000, "C": C,
                       "warn_folds": n_warn, "seconds": round(secs, 1),
                       "roc_auc": roc_auc_score(y, s),
                       "pr_auc": average_precision_score(y, s),
                       "tuned_cost": tuned["aps_cost"],
                       "honest_cost": h_cost, "honest_fp": h_fp, "honest_fn": h_fn})
    print(f"C={C} g={g} -> {h_cost} ({secs:.0f}s, warn={n_warn})")

pd.DataFrame(combo_rows).sort_values("honest_cost").round(4)

C=1.0 g=0.1 -> 48070 (177s, warn=0)
C=1.0 g=0.03 -> 48090 (164s, warn=0)
C=10.0 g=0.03 -> 47440 (389s, warn=0)


,model,gamma_mult,n_components,C,warn_folds,seconds,roc_auc,pr_auc,tuned_cost,honest_cost,honest_fp,honest_fn
2,Linear+RBF-Nystroem,0.03,1000,10.0,0,388.6,0.9848,0.8343,46420,47440,1744,60
0,Linear+RBF-Nystroem,0.10,1000,1.0,0,176.8,0.9855,0.8379,45620,48070,2157,53
1,Linear+RBF-Nystroem,0.03,1000,1.0,0,163.9,0.9855,0.8375,45360,48090,2009,56


In [13]:
RES = PROJECT_ROOT / "reports" / "results"
RES.mkdir(parents=True, exist_ok=True)
print("already saved:", sorted(p.name for p in RES.glob("phase5_*.csv")))

# --- fold-by-fold honest costs (transcribed from the printed table) ---
fold_costs_df = pd.DataFrame(
    {"SVM C=1 unweighted": [8920, 10650, 8480, 8420, 8630],
     "LogReg unweighted":  [8050, 10830, 9660, 10030, 10180],
     "LogReg balanced":    [9820, 12070, 9350, 11100, 9590]},
    index=[f"fold {i}" for i in range(1, 6)])
fold_costs_df.loc["total"] = fold_costs_df.sum()

# --- linear SVM grid (C, class_weight, seconds, roc, pr, tuned, honest, fp, fn) ---
lin = [
 (1.0,   None,       57.2, 0.9855, 0.8371, 44640, 45100, 2110, 48),
 (0.003, "balanced", 148.0, 0.9862, 0.7772, 44980, 45450, 2245, 46),
 (30.0,  None,       212.6, 0.9851, 0.8345, 45660, 46110, 1811, 56),
 (100.0, None,       160.8, 0.9850, 0.8342, 45660, 46150, 1815, 56),
 (0.1,   None,       51.0, 0.9860, 0.8367, 45710, 46180, 1868, 55),
 (10.0,  None,       117.3, 0.9851, 0.8350, 45760, 47180, 1968, 55),
 (3.0,   None,       55.1, 0.9853, 0.8361, 45930, 48570, 2257, 52),
 (0.001, "balanced", 151.7, 0.9860, 0.7630, 45570, 49130, 2713, 44),
 (0.01,  "balanced", 77.1, 0.9859, 0.7842, 47310, 50040, 2104, 58),
 (0.1,   "balanced", 80.2, 0.9838, 0.7807, 50270, 53830, 2033, 67),
 (0.01,  None,       37.9, 0.9850, 0.8155, 50470, 54580, 2008, 69),
 (1.0,   "balanced", 129.6, 0.9827, 0.7747, 52050, 54640, 2114, 67),
]
tail = ["seconds", "roc_auc", "pr_auc", "tuned_cost", "honest_cost", "honest_fp", "honest_fn"]
svm_linear_results = pd.DataFrame(lin, columns=["C", "class_weight"] + tail)
svm_linear_results.insert(0, "model", "LinearSVC")
svm_linear_results.insert(4, "warn_folds", 0)

# --- RBF-Nystroem grid (gamma_mult, n_components, C, seconds, roc, pr, tuned, honest, fp, fn) ---
rbf = [
 (0.03, 1000, 10.0, 163.6, 0.9846, 0.8061, 46280, 47550, 2005, 55),
 (0.10, 1000, 1.0,  84.2,  0.9840, 0.7765, 47920, 48780, 1978, 58),
 (0.10, 1000, 10.0, 122.1, 0.9828, 0.8126, 49230, 50990, 1899, 64),
 (0.03, 1000, 1.0,  87.7,  0.9827, 0.7482, 51710, 53090, 2959, 47),
 (0.30, 1000, 1.0,  117.3, 0.9824, 0.7691, 52150, 54520, 2252, 64),
 (0.10, 300,  1.0,  56.0,  0.9831, 0.7482, 53370, 55300, 2280, 65),
 (0.03, 300,  1.0,  60.0,  0.9818, 0.7346, 55940, 57600, 3360, 48),
 (1.00, 1000, 1.0,  178.0, 0.9757, 0.6701, 58550, 59050, 2555, 67),
 (0.01, 1000, 1.0,  88.5,  0.9798, 0.7125, 64260, 66950, 3895, 56),
 (0.01, 300,  1.0,  42.8,  0.9793, 0.7073, 65730, 69600, 3810, 63),
]
rbf_results = pd.DataFrame(rbf, columns=["gamma_mult", "n_components", "C"] + tail)
rbf_results.insert(0, "model", "RBF-Nystroem+LinearSVC")
rbf_results.insert(4, "warn_folds", 0)

# --- Linear + RBF combined (gamma_mult, C, seconds, roc, pr, tuned, honest, fp, fn) ---
cmb = [
 (0.03, 10.0, 388.6, 0.9848, 0.8343, 46420, 47440, 1744, 60),
 (0.10, 1.0,  176.8, 0.9855, 0.8379, 45620, 48070, 2157, 53),
 (0.03, 1.0,  163.9, 0.9855, 0.8375, 45360, 48090, 2009, 56),
]
combo_results = pd.DataFrame(cmb, columns=["gamma_mult", "C"] + tail)
combo_results.insert(0, "model", "Linear+RBF-Nystroem")
combo_results.insert(3, "n_components", 1000)
combo_results.insert(5, "warn_folds", 0)

# integrity check: every row must satisfy cost = 10*FP + 500*FN
for name, df in [("linear", svm_linear_results), ("rbf", rbf_results), ("combo", combo_results)]:
    assert (10 * df.honest_fp + 500 * df.honest_fn == df.honest_cost).all(), f"typo in {name} table"
print("all rows pass the cost check")

def save_if_missing(df, name, **kw):
    path = RES / name
    if path.exists():
        print("kept existing:", name)
    else:
        df.to_csv(path, **kw)
        print("saved:", name)

save_if_missing(fold_costs_df,       "phase5_fold_costs.csv")
save_if_missing(svm_linear_results,  "phase5_linear_svm_results.csv", index=False)
save_if_missing(rbf_results,         "phase5_rbf_nystroem_results.csv", index=False)
save_if_missing(combo_results,       "phase5_linear_plus_rbf_results.csv", index=False)

already saved: []
all rows pass the cost check
saved: phase5_fold_costs.csv
saved: phase5_linear_svm_results.csv
saved: phase5_rbf_nystroem_results.csv
saved: phase5_linear_plus_rbf_results.csv


In [14]:
summary = pd.DataFrame([
    {"model": "Do nothing",              "honest_cost": 500_000},
    {"model": "LogReg balanced + log",   "honest_cost": int(fold_costs_df.loc["total", "LogReg balanced"])},
    {"model": "LogReg unweighted + log", "honest_cost": int(fold_costs_df.loc["total", "LogReg unweighted"])},
    {"model": "RBF-Nystroem (best)",     "honest_cost": int(rbf_results["honest_cost"].min())},
    {"model": "Linear + RBF (best)",     "honest_cost": int(combo_results["honest_cost"].min())},
    {"model": "Linear SVM (best)",       "honest_cost": int(svm_linear_results["honest_cost"].min())},
]).sort_values("honest_cost").reset_index(drop=True)

summary["cost_per_truck"] = (summary["honest_cost"] / 60_000).round(3)
summary["saved_vs_do_nothing_%"] = ((1 - summary["honest_cost"] / 500_000) * 100).round(1)

save_if_missing(summary, "phase5_summary.csv", index=False)
summary

saved: phase5_summary.csv


,model,honest_cost,cost_per_truck,saved_vs_do_nothing_%
0,Linear SVM (best),45100,0.752,91.0
1,Linear + RBF (best),47440,0.791,90.5
2,RBF-Nystroem (best),47550,0.792,90.5
3,LogReg unweighted + log,48750,0.812,90.2
4,LogReg balanced + log,51930,0.866,89.6
5,Do nothing,500000,8.333,0.0


## Phase 5 conclusion

**Result.** The best model in Phase 5 was the linear SVM (C = 1, unweighted), with an
honest out-of-fold cost of **45,100** (about 0.75 per truck, 91.0% below the
500,000 cost of flagging nothing). The best Logistic Regression (unweighted, with the
signed-log transform) cost 48,750, so the SVM was about 7.5% cheaper; against balanced
Logistic Regression (51,930) it was about 13% cheaper. The SVM had the lowest cost in
5 of 5 folds against the balanced model and 4 of 5 against the unweighted one.

**Kernel models.** The RBF-Nystroem models (best 47,550) and the combined
linear + RBF models (best 47,440) did not beat the linear SVM. Their ranking quality was
essentially the same (ROC-AUC about 0.985, PR-AUC about 0.83), and the extra cost of
the combined models came with a larger gap between tuned and honest cost, which
suggests a less stable threshold (a hypothesis, not tested). Together with the fact that
the best RBF settings were the ones closest to a linear model, this suggests there is
little curved structure that this approximation can capture. It does not prove there is none.

**Class weighting.** It did not help either model family. Unweighted Logistic Regression
beat balanced Logistic Regression, and the balanced SVM only matched the unweighted one
at very strong regularization (C = 0.003: 45,450 vs 45,100).

**Caveats.**
- Only 5 folds, and individual folds vary by thousands (about 8,000 to 12,000), so the SVM's
  edge over unweighted Logistic Regression (3,650 in total) is promising but not statistically established.
- The SVM was the best of many tried settings, chosen on the same out-of-fold predictions,
  while Logistic Regression was run at default settings with no tuning of C. This favors the SVM.
- Nystroem is an approximation (random landmarks, 300 to 1000), and the grid was coarse.
- Thresholds were tuned on out-of-fold scores; the fold-wise "honest" estimate is close
  to, but not strictly, leak-free.
- The RBF and combined models have no fold-by-fold comparison, because their scores were not saved.
- Costs are in dataset units, not real money. The official test set has not been touched,
  so these results make no claim about test performance.